# Домашнє завдання: Внесення оновлень в БД і робота з транзакціями

Це ДЗ передбачене під виконання на локальній машині. Виконання з Google Colab буде суттєво ускладнене.

## Підготовка
1. Переконайтесь, що у вас встановлены необхідні бібліотеки:
   ```bash
   pip install sqlalchemy pymysql pandas matplotlib seaborn python-dotenv
   ```

2. Створіть файл `.env` з параметрами підключення до бази даних classicmodels. Базу даних ви можете отримати через

  - docker-контейнер згідно існтрукції в [документі](https://www.notion.so/hannapylieva/Docker-1eb94835849480c9b2e7f5dc22ee4df9), також відео інструкції присутні на платформі - уроки "MySQL бази, клієнт для роботи з БД, Docker і ChatGPT для запитів" та "Як встановити Docker для роботи з базами даних без терміналу"
  - або встановивши локально цю БД - для цього перегляньте урок "Опціонально. Встановлення MySQL та  БД Сlassicmodels локально".
  
  Приклад `.env` файлу ми створювали в лекції. Ось його обовʼязкове наповнення:
    ```
    DB_HOST=your_host
    DB_PORT=3306 або 3307 - той, який Ви налаштували
    DB_USER=your_username
    DB_PASSWORD=your_password
    DB_NAME=classicmodels
    ```
  Якщо ви створили цей файл під час перегляду лекції - **новий створювати не треба**. Замініть лише назву БД, або пропишіть назву в коді створення підключення (замість отримання назви цільової БД зі змінних оточення). Але переконайтесь, що до `.env` файл лежить в тій самій папці, що і цей ноутбук.

  **УВАГА!** НЕ копіюйте скрит для **створення** `.env` файлу. В лекції він наводиться для прикладу. І давалось пояснення, що в реальних проєктах ми НІКОЛИ не пишемо доступи до бази в коді. Копіювання скрипта для створення `.env` файлу сюди в ДЗ буде вважатись грубою помилкою і ми зніматимемо бали.

3. Налаштуйте підключення через SQLAlchemy до БД за прикладом в лекції.

Рекомендую вивести (відобразити) змінну engine після створення. Вона має бути не None! Якщо None - значить у Вас не підтягнулись налаштування з .env файла.

Ви також можете налаштувати параметри підключення до БД без .env файла, просто прописавши текстом в відповідних місцях. Це - не рекомендований підхід.


## Завдання

### Завдання 1: Оновлення інформації про клієнта (2 бали)

**Створіть функцію для оновлення контактної інформації клієнта за його номером** з наступними можливостями:
- Оновлення телефону клієнта
- Оновлення email (якщо поле існує в таблиці)

Опціонально, якщо вам хочеться більше практики:
- Логування змін в окрему таблицю

Використайте підхід з параметризованими запитами через `text()` та `UPDATE` оператор. Не забудьте на початку перевірити чи існує клієнт з таким номером в базі - це хороша практика.

Отримати всі колонки, які існують в таблиці ви можете наступним запитом
```
  SELECT COLUMN_NAME, DATA_TYPE
  FROM INFORMATION_SCHEMA.COLUMNS
  WHERE TABLE_NAME = 'customers'
```

Запустіть функцію і продемонструйте її роботу, запустивши SELECT, який допоможе це зробити.



## Підготовка :

In [1]:
# Перевіряємо наявність потрібних бібліотек:
!pip list | findstr "SQLAlchemy PyMySQL pandas matplotlib seaborn python-dotenv"

matplotlib                        3.10.6
matplotlib-inline                 0.2.1
pandas                            2.3.3
PyMySQL                           1.2.3
python-dotenv                     1.1.0
seaborn                           0.13.2
SQLAlchemy                        2.0.43


In [32]:
# Імпортуємо необхідні бібліотеки
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import datetime   

In [3]:
# Завантажуємо параметри підключення з файлу .env
load_dotenv(override=True)

True

In [4]:
# Створюємо підключення до БД через SQLAlchemy
def create_connection():
    host = os.getenv("DB_HOST")
    port = os.getenv("DB_PORT")
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")
    database = os.getenv("DB_NAME")

    # Створюємо рядок підключення
    connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"

    # Створюємо engine
    engine = create_engine(
        connection_string,
        pool_size=2,
        max_overflow=20,
        pool_pre_ping=True,
        echo=False)

    # Перевіряємо підключення
    try:
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))

        print("Підключення до БД успішне!")
        print(f"Engine створено: {engine is not None}")

        return engine

    except Exception as e:
        print(f"Помилка підключення: {e}")
        return None


# Створюємо підключення
engine = create_connection()

Підключення до БД успішне!
Engine створено: True


In [5]:
# Перевіряємо, до якої бази даних виконано підключення
with engine.connect() as conn:
    result = conn.execute(text("SELECT DATABASE()"))
    print(result.scalar())

classicmodels


### Завдання 1: Оновлення інформації про клієнта (2 бали)

**Створіть функцію для оновлення контактної інформації клієнта за його номером** з наступними можливостями:
- Оновлення телефону клієнта
- Оновлення email (якщо поле існує в таблиці)

Опціонально, якщо вам хочеться більше практики:
- Логування змін в окрему таблицю

Використайте підхід з параметризованими запитами через `text()` та `UPDATE` оператор. Не забудьте на початку перевірити чи існує клієнт з таким номером в базі - це хороша практика.

Отримати всі колонки, які існують в таблиці ви можете наступним запитом
```
  SELECT COLUMN_NAME, DATA_TYPE
  FROM INFORMATION_SCHEMA.COLUMNS
  WHERE TABLE_NAME = 'customers'
```

Запустіть функцію і продемонструйте її роботу, запустивши SELECT, який допоможе це зробити.


In [16]:
# Перевіряємо структуру таблиці customers
query = text("""
SELECT COLUMN_NAME, DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = DATABASE()
  AND TABLE_NAME = 'customers'
""")

pd.read_sql(query, engine)

,COLUMN_NAME,DATA_TYPE
0,customerNumber,int
1,customerName,varchar
2,contactLastName,varchar
3,contactFirstName,varchar
4,phone,varchar
5,addressLine1,varchar
6,addressLine2,varchar
7,city,varchar
8,state,varchar
9,postalCode,varchar


**У таблиці customers поле email відсутнє, тому функція виконуватиме оновлення лише номера телефону.**

In [17]:
# Переглядаємо клієнтів перед оновленням
query = text("""
SELECT customerNumber, customerName, phone
FROM customers
LIMIT 5
""")

pd.read_sql(query, engine)

,customerNumber,customerName,phone
0,103,Atelier graphique,+380997774422
1,112,Signal Gift Stores,7025551838
2,114,"Australian Collectors, Co.",03 9520 4555
3,119,La Rochelle Gifts,40.67.8555
4,121,Baane Mini Imports,07-98 9555


In [18]:
create_log_table = text("""                             # Формуємо SQL-запит для створення таблиці логування
CREATE TABLE IF NOT EXISTS customer_changes_log (       # Створюємо таблицю customer_changes_log якщо вона ще не існує
    id INT AUTO_INCREMENT PRIMARY KEY,                  # Поле id: ціле число, первинний ключ, значення збільшується автоматично
    customerNumber INT,                                 # Номер клієнта: тип ціле числове значення
    old_phone VARCHAR(50),                              # Старий телефон: VARCHAR(50)  текстове поле довжиною до 50 символів
    new_phone VARCHAR(50),                              # Новий телефон: VARCHAR(50)  текстове поле довжиною до 50 символів
    changed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP      # Дата і час зміни: TIMESTAMP автоматично отримує поточну дату і час
)
""")
with engine.connect() as conn:                          # Відкриваємо підключення до бази даних
    conn.execute(create_log_table)                      # Виконуємо SQL-запит на створення таблиці
    conn.commit()                                       # Підтверджуємо транзакцію і зберігаємо зміни в БД
print("Таблицю для логування створено")                 # Виводимо повідомлення про успішне створення таблиці


Таблицю для логування створено


In [19]:
# Перевіряємо список таблиць у базі даних
query = text("""
SHOW TABLES
""")

pd.read_sql(query, engine)

,Tables_in_classicmodels
0,customer_changes_log
1,customers
2,employees
3,offices
4,orderdetails
5,orders
6,payments
7,productlines
8,products


In [20]:
# Функція для оновлення телефону клієнта з логуванням змін
# та захистом від повторного оновлення тим самим номером

def update_customer_phone(customer_number, new_phone):

    # Перевіряємо, чи існує клієнт, та отримуємо його поточний телефон
    check_query = text("""
        SELECT phone
        FROM customers
        WHERE customerNumber = :customer_number
    """)

    # Оновлюємо телефон клієнта
    update_query = text("""
        UPDATE customers
        SET phone = :new_phone
        WHERE customerNumber = :customer_number
    """)

    # Записуємо зміну в таблицю логів
    log_query = text("""
        INSERT INTO customer_changes_log
            (customerNumber, old_phone, new_phone)
        VALUES
            (:customer_number, :old_phone, :new_phone)
    """)

    with engine.connect() as conn:

        # Шукаємо клієнта
        customer = conn.execute(
            check_query,
            {"customer_number": customer_number}
        ).fetchone()

        # Якщо клієнта немає — завершуємо функцію
        if customer is None:
            print("Клієнта з таким номером не знайдено")
            return False

        # Зберігаємо поточний номер телефону
        old_phone = customer[0]

        # Перевіряємо, чи телефон дійсно змінився
        if old_phone == new_phone:
            print(
                "Новий номер телефону співпадає з поточним. "
                "Оновлення не потрібне."
            )
            return False

        # Оновлюємо телефон
        conn.execute(
            update_query,
            {
                "new_phone": new_phone,
                "customer_number": customer_number
            }
        )

        # Записуємо зміну в журнал
        conn.execute(
            log_query,
            {
                "customer_number": customer_number,
                "old_phone": old_phone,
                "new_phone": new_phone
            }
        )

        # Підтверджуємо зміни
        conn.commit()

    print("Телефон клієнта успішно оновлено")
    print("Зміну записано в журнал")

    return True

**Запускаємо функцію:**

In [29]:
# для клієнта 103
update_customer_phone(103, "+15551234567")

Телефон клієнта успішно оновлено
Зміну записано в журнал


True

In [30]:
# Перевіряємо результат оновлення
query = text("""
SELECT customerNumber, customerName, phone
FROM customers
WHERE customerNumber = 103
""")

pd.read_sql(query, engine)

,customerNumber,customerName,phone
0,103,Atelier graphique,+15551234567


In [31]:
# Перевіряємо журнал змін
query = text("""
SELECT *
FROM customer_changes_log
WHERE customerNumber = 103
ORDER BY changed_at DESC
""")

pd.read_sql(query, engine)

,id,customerNumber,old_phone,new_phone,changed_at
0,8,103,+380997775555,+15551234567,2026-09-24 16:26:11
1,7,103,+380997774422,+380997775555,2026-09-24 16:15:17
2,6,103,+380671234567,+380997774422,2026-09-24 15:52:59
3,5,103,+380971234567,+380671234567,2026-09-24 04:12:11
4,4,103,+380971234567,+380671234567,2026-09-19 18:19:19
5,3,103,+380671234567,+380971234567,2026-09-19 16:53:39
6,2,103,+380671234567,+380671234567,2026-09-19 16:52:35
7,1,103,+380501234567,+380671234567,2026-09-19 16:36:22


### Завдання 2: Створення нового замовлення з транзакцією (5 балів)

**Реалізуйте процес створення нового замовлення** з наступними кроками в одній транзакції:
- Створення запису в таблиці `orders`
- Додавання товарних позицій в `orderdetails`
- Перевірка наявності товарів на складі
- Зменшення кількості товарів на складі

Запустіть процес з тестовими даними і продемонструйте через SELECT, що процес успішно відпрацював і були виконані необхідні операції.




In [67]:
# Перевіряємо структуру таблиці orders
query = text("""
DESCRIBE orders
""")

pd.read_sql(query, engine)

,Field,Type,Null,Key,Default,Extra
0,orderNumber,int,NO,,None,
1,orderDate,date,NO,,None,
2,requiredDate,date,NO,,None,
3,shippedDate,date,YES,,None,
4,status,varchar(15),YES,,None,
5,comments,text,YES,,None,
6,customerNumber,int,NO,,None,


In [68]:
# Перевіряємо структуру таблиці orderdetails
query = text("""
DESCRIBE orderdetails
""")

pd.read_sql(query, engine)

,Field,Type,Null,Key,Default,Extra
0,orderNumber,int,NO,,None,
1,productCode,varchar(15),YES,,None,
2,quantityOrdered,int,NO,,None,
3,priceEach,"decimal(10,2)",YES,,None,
4,orderLineNumber,smallint,NO,,None,


In [69]:
# Перевіряємо структуру таблиці products
query = text("""
DESCRIBE products
""")

pd.read_sql(query, engine)

,Field,Type,Null,Key,Default,Extra
0,productCode,varchar(15),YES,,None,
1,productName,varchar(70),YES,,None,
2,productLine,varchar(50),YES,,None,
3,productScale,varchar(10),YES,,None,
4,productVendor,varchar(50),YES,,None,
5,productDescription,text,NO,,None,
6,quantityInStock,smallint,NO,,None,
7,buyPrice,"decimal(10,2)",YES,,None,
8,MSRP,"decimal(10,2)",YES,,None,


In [70]:
# Переглядаємо клієнтів для створення тестового замовлення
query = text("""
SELECT customerNumber,        
       customerName           
FROM customers
LIMIT 10
""")

pd.read_sql(query, engine)

,customerNumber,customerName
0,103,Atelier graphique
1,112,Signal Gift Stores
2,114,"Australian Collectors, Co."
3,119,La Rochelle Gifts
4,121,Baane Mini Imports
5,124,Mini Gifts Distributors Ltd.
6,125,Havel & Zbyszek Co
7,128,"Blauer See Auto, Co."
8,129,Mini Wheels Co.
9,131,Land of Toys Inc.


In [71]:
# Визначаємо новий номер замовлення

query = text("""
SELECT MAX(orderNumber) AS max_order_number
FROM orders
""")

max_order = pd.read_sql(query, engine)

new_order_number = int(max_order.loc[0, "max_order_number"]) + 1

print("Новий номер замовлення:", new_order_number)

Новий номер замовлення: 10428


*Перед створенням транзакції зафіксуємо всі тестові дані в окремій комірці:*

In [72]:
customer_number = 103             # Номер клієнта Atelier graphique

order_items = [                   # Список товарних позицій нового замовлення
    ("S10_1678", 5, 95.70),       # Код товару, кількість 5 шт., ціна за одиницю 95.70
    ("S10_1949", 10, 214.30),      # Код товару, кількість 10 шт., ціна за одиницю 214.30
    ("S10_2016", 7, 118.94)       # Код товару, кількість 7 шт., ціна за одиницю 118.94
]

In [73]:
# Зберігаємо залишки товарів до виконання транзакції

query = text("""
SELECT
    productCode,
    productName,
    quantityInStock
FROM products
WHERE productCode IN ('S10_1678', 'S10_1949', 'S10_2016')
ORDER BY productCode
""")

stock_before = pd.read_sql(query, engine)

stock_before

,productCode,productName,quantityInStock
0,S10_1678,1969 Harley Davidson Ultimate Chopper,7929
1,S10_1949,1952 Alpine Renault 1300,7299
2,S10_2016,1996 Moto Guzzi 1100i,6623


In [74]:
def create_order_with_transaction(engine, order_number, customer_number, items):
    """Створення нового замовлення в одній транзакції без дублювання"""

    today = datetime.date.today()
    required_date = today + datetime.timedelta(days=7)

    # Перевіряємо, чи замовлення з таким номером вже існує
    check_order_query = text("""
        SELECT orderNumber
        FROM orders
        WHERE orderNumber = :order_number
    """)

    # Перевіряємо існування клієнта
    check_customer_query = text("""
        SELECT customerNumber
        FROM customers
        WHERE customerNumber = :customer_number
    """)

    # Створюємо нове замовлення
    insert_order_query = text("""
        INSERT INTO orders (
            orderNumber,
            orderDate,
            requiredDate,
            status,
            comments,
            customerNumber
        )
        VALUES (
            :order_number,
            :order_date,
            :required_date,
            :status,
            :comments,
            :customer_number
        )
    """)

    # Перевіряємо залишок товару
    check_stock_query = text("""
        SELECT quantityInStock
        FROM products
        WHERE productCode = :product_code
    """)

    # Додаємо товарну позицію до замовлення
    insert_detail_query = text("""
        INSERT INTO orderdetails (
            orderNumber,
            productCode,
            quantityOrdered,
            priceEach,
            orderLineNumber
        )
        VALUES (
            :order_number,
            :product_code,
            :quantity,
            :price,
            :line_number
        )
    """)

    # Зменшуємо залишок товару на складі
    update_stock_query = text("""
        UPDATE products
        SET quantityInStock = quantityInStock - :quantity
        WHERE productCode = :product_code
    """)

    try:
        # Відкриваємо одну транзакцію
        with engine.begin() as conn:

            # КРОК 0. Перевіряємо, чи замовлення вже існує
            existing_order = conn.execute(
                check_order_query,
                {"order_number": order_number}
            ).fetchone()

            if existing_order is not None:
                print(
                    f"Замовлення {order_number} вже існує. "
                    "Повторне створення скасовано."
                )
                return False

            # КРОК 1. Перевіряємо клієнта
            customer = conn.execute(
                check_customer_query,
                {"customer_number": customer_number}
            ).fetchone()

            if customer is None:
                raise ValueError(
                    f"Клієнта {customer_number} не знайдено"
                )

            # Створюємо нове замовлення
            conn.execute(
                insert_order_query,
                {
                    "order_number": order_number,
                    "order_date": today,
                    "required_date": required_date,
                    "status": "In Process",
                    "comments": "Test order HW_12_2",
                    "customer_number": customer_number
                }
            )

            print(f"Крок 1: Замовлення {order_number} створено")

            # Обробляємо всі товарні позиції
            for line_number, (product_code, quantity, price) in enumerate(
                items, start=1
            ):

                # Перевіряємо наявність та залишок товару
                stock = conn.execute(
                    check_stock_query,
                    {"product_code": product_code}
                ).fetchone()

                if stock is None:
                    raise ValueError(
                        f"Товар {product_code} не знайдено"
                    )

                quantity_in_stock = stock[0]

                if quantity_in_stock < quantity:
                    raise ValueError(
                        f"Недостатньо товару {product_code}. "
                        f"На складі: {quantity_in_stock}, "
                        f"потрібно: {quantity}"
                    )

                print(
                    f"Крок 2: {product_code} — "
                    f"на складі {quantity_in_stock}, "
                    f"потрібно {quantity}"
                )

                # Додаємо товар до orderdetails
                conn.execute(
                    insert_detail_query,
                    {
                        "order_number": order_number,
                        "product_code": product_code,
                        "quantity": quantity,
                        "price": price,
                        "line_number": line_number
                    }
                )

                # Зменшуємо залишок товару
                conn.execute(
                    update_stock_query,
                    {
                        "quantity": quantity,
                        "product_code": product_code
                    }
                )

                print(
                    f"Крок 3: {product_code} додано до замовлення, "
                    f"залишок зменшено на {quantity}"
                )

        print("Всі операції виконано успішно — COMMIT")
        return True

    except Exception as e:
        print(f"Помилка: {e}")
        print("Всі зміни скасовано — ROLLBACK")
        return False

**Запускаємо функцію:**

In [75]:
# Запускаємо функцію створення нового замовлення в транзакції

create_order_with_transaction(
    engine,                    # Підключення до бази даних
    new_order_number,          # Номер нового замовлення 10428
    customer_number,           # Номер клієнта 103
    order_items                # Список із трьох товарних позицій
)

Крок 1: Замовлення 10428 створено
Крок 2: S10_1678 — на складі 7929, потрібно 5
Крок 3: S10_1678 додано до замовлення, залишок зменшено на 5
Крок 2: S10_1949 — на складі 7299, потрібно 10
Крок 3: S10_1949 додано до замовлення, залишок зменшено на 10
Крок 2: S10_2016 — на складі 6623, потрібно 7
Крок 3: S10_2016 додано до замовлення, залишок зменшено на 7
Всі операції виконано успішно — COMMIT


True

In [76]:
# Зберігаємо залишки трьох товарів після виконання транзакції

query = text("""
SELECT productCode,
       productName,
       quantityInStock
FROM products
WHERE productCode IN ('S10_1678', 'S10_1949', 'S10_2016')
""")

stock_after = pd.read_sql(query, engine)

stock_after

,productCode,productName,quantityInStock
0,S10_1678,1969 Harley Davidson Ultimate Chopper,7924
1,S10_1949,1952 Alpine Renault 1300,7289
2,S10_2016,1996 Moto Guzzi 1100i,6616


**Перевірка:**

In [77]:
# Перевіряємо створене замовлення в таблиці orders
query = text("""
SELECT *
FROM orders
WHERE orderNumber = 10428
""")

pd.read_sql(query, engine)

,orderNumber,orderDate,requiredDate,shippedDate,status,comments,customerNumber
0,10428,2026-09-24,2026-10-01,None,In Process,Test order HW_12_2,103


In [78]:
# Перевіряємо товарні позиції нового замовлення
query = text("""
SELECT *
FROM orderdetails
WHERE orderNumber = 10428
ORDER BY orderLineNumber
""")

pd.read_sql(query, engine)

,orderNumber,productCode,quantityOrdered,priceEach,orderLineNumber
0,10428,S10_1678,5,95.70,1
1,10428,S10_1949,10,214.30,2
2,10428,S10_2016,7,118.94,3


In [79]:
# Порівнюємо залишки товарів до і після виконання транзакції

comparison = stock_before.merge(
    stock_after,
    on=["productCode", "productName"],
    suffixes=("_before", "_after")
)

comparison["difference"] = (
    comparison["quantityInStock_after"]
    - comparison["quantityInStock_before"]
)

comparison

,productCode,productName,quantityInStock_before,quantityInStock_after,difference
0,S10_1678,1969 Harley Davidson Ultimate Chopper,7929,7924,-5
1,S10_1949,1952 Alpine Renault 1300,7299,7289,-10
2,S10_2016,1996 Moto Guzzi 1100i,6623,6616,-7
